In [59]:
import kagglehub
path = kagglehub.dataset_download("annavictoria/speed-dating-experiment")
import pandas as pd
file_path = path + "/Speed Dating Data.csv"
df = pd.read_csv(file_path, encoding='latin-1')

In [60]:
df.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Data columns (total 195 columns):
 #    Column    Non-Null Count  Dtype  
---   ------    --------------  -----  
 0    iid       8378 non-null   int64  
 1    id        8377 non-null   float64
 2    gender    8378 non-null   int64  
 3    idg       8378 non-null   int64  
 4    condtn    8378 non-null   int64  
 5    wave      8378 non-null   int64  
 6    round     8378 non-null   int64  
 7    position  8378 non-null   int64  
 8    positin1  6532 non-null   float64
 9    order     8378 non-null   int64  
 10   partner   8378 non-null   int64  
 11   pid       8368 non-null   float64
 12   match     8378 non-null   int64  
 13   int_corr  8220 non-null   float64
 14   samerace  8378 non-null   int64  
 15   age_o     8274 non-null   float64
 16   race_o    8305 non-null   float64
 17   pf_o_att  8289 non-null   float64
 18   pf_o_sin  8289 non-null   float64
 19   pf_o_int  8289 non-null   float64
 20   pf_o_f

In [61]:
#missing id can be derived from other identifying variables
df.loc[8377, 'id'] = 22.0
#idg and id columns are have swapped values - idg should have unique id within gender, and id within wave but it is other way around
df.rename(columns={'id': 'idg', 'idg': 'id'}, inplace=True)
#some pid are missing easy to derive what is supposed to be there - in wave 5 only 118 is not partner in any date
df.loc[df['pid'].isnull(), 'pid'] = 118.0
#we can drop positions as it has no practical use
df = df.drop(['position', 'positin1'],axis='columns')
#int_corr can be calculated later

#age_o and race_o, all pf_o_(attribute) - stated pref,  some can be derived form pid rest can be dropped since there is little of them
#how partner answered their scoreboard about subject probably can be derived from other records

#fields requires some minor tweak, rest can be imputed with mode
df.loc[df['field'] == 'Operations Research', 'field_cd'] = 5.0
#undergrd maybe have more than 50% notna but we dont really care about school they attended
df = df.drop('undergra',axis='columns')
#imprace and imprelig impute with median
#from, impute wit mode/'other'

#assessing income by zipcode is not reliable and only about 50% was assigned income, we drop zipcode and income
df = df.drop(['zipcode', 'income'],axis='columns')
#goal date and go_out not many missing, impute with median

#career some tweak rest impute with 'other'
df.loc[(df['career'] == 'lawyer') | (df['career'] == 'law'), 'career_c'] = 1.0
df.loc[df['career'] == 'Economist', 'career_c'] = 7.0
df.loc[df['career'] == 'tech professional', 'career_c'] = 5.0
#Lets impute missing careers and fields with other category
df.loc[df['career'].isna(), 'career_c'] = 15.0
df.loc[df['field'].isna(), 'field_cd'] = 18.0


#sports-yoga few missing, impute with median

#exphappy not really important
df = df.drop('exphappy',axis='columns')

#like prob met - few missing impute with median

#match_es validate against wave participants, rather drop if nans if needed in further research

#all of attributes ratings can be dealt when needed
#those during actual date are mostly there so impute with median
#those gathered after speed dating events many missing but we can use only those when researching further

#you_call, them_call, date_3 we can drop nans in batch for this analysis
#numdat_3 we can impute some nans with 0 if date_3 == 0, rest if date_3 == 1 impute with median



In [62]:
#in columns regarding attributes some do not sum up to 100 like in this example, we will normalize it later so it all will add up to 100
#waves 6-9 are already normalized to 100 distribution point scale
suffix = '1_1'
cols = [f"{prefix}{suffix}" for prefix in ['attr', 'sinc', 'intel', 'fun', 'amb', 'shar']]
total_points = df[cols].sum(axis=1, skipna=False)
total_points = total_points.round(0)
print("Distribution of sums:")
print(total_points.value_counts().head(10))
nan_rows_count = total_points.isna().sum()
print(f"\nNumber of rows with missing (NaN) data: {nan_rows_count}")
bad_rows= (total_points != 100) & (total_points.notnull())
print(f"Number of rows where the sum is NOT 100: {bad_rows.sum()}")

Distribution of sums:
100.0    8103
90.0       65
120.0      25
95.0       22
110.0      22
148.0      10
101.0      10
Name: count, dtype: int64

Number of rows with missing (NaN) data: 121
Number of rows where the sum is NOT 100: 154


In [63]:
missing = df.isnull().mean()
columns_to_drop = missing[missing >= 0.50].index
print("Columns that are 50%+ empty:")
print(list(columns_to_drop))
#we will be dropping these columns that have too much missing data
df = df.drop(['mn_sat', 'tuition', 'expnum'],axis='columns')
#with other mostly missing columns we can subset whose that are not missing to analyze them specifically
#e.g. there is column to specify how many dates one got after event, so if someone didnt get them they left it empty

Columns that are 50%+ empty:
['mn_sat', 'tuition', 'expnum', 'attr1_s', 'sinc1_s', 'intel1_s', 'fun1_s', 'amb1_s', 'shar1_s', 'attr3_s', 'sinc3_s', 'intel3_s', 'fun3_s', 'amb3_s', 'attr7_2', 'sinc7_2', 'intel7_2', 'fun7_2', 'amb7_2', 'shar7_2', 'you_call', 'them_cal', 'date_3', 'numdat_3', 'num_in_3', 'attr1_3', 'sinc1_3', 'intel1_3', 'fun1_3', 'amb1_3', 'shar1_3', 'attr7_3', 'sinc7_3', 'intel7_3', 'fun7_3', 'amb7_3', 'shar7_3', 'attr4_3', 'sinc4_3', 'intel4_3', 'fun4_3', 'amb4_3', 'shar4_3', 'attr2_3', 'sinc2_3', 'intel2_3', 'fun2_3', 'amb2_3', 'shar2_3', 'attr3_3', 'sinc3_3', 'intel3_3', 'fun3_3', 'amb3_3', 'attr5_3', 'sinc5_3', 'intel5_3', 'fun5_3', 'amb5_3']


In [64]:
#we can drop rows(there is only few of them) where people didnt filled in the introductory list 
df[df['attr1_1'].isna()]
df = df[df['attr1_1'].notna()]
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 8299 entries, 0 to 8377
Data columns (total 186 columns):
 #    Column    Non-Null Count  Dtype  
---   ------    --------------  -----  
 0    iid       8299 non-null   int64  
 1    idg       8299 non-null   float64
 2    gender    8299 non-null   int64  
 3    id        8299 non-null   int64  
 4    condtn    8299 non-null   int64  
 5    wave      8299 non-null   int64  
 6    round     8299 non-null   int64  
 7    order     8299 non-null   int64  
 8    partner   8299 non-null   int64  
 9    pid       8299 non-null   float64
 10   match     8299 non-null   int64  
 11   int_corr  8220 non-null   float64
 12   samerace  8299 non-null   int64  
 13   age_o     8195 non-null   float64
 14   race_o    8226 non-null   float64
 15   pf_o_att  8210 non-null   float64
 16   pf_o_sin  8210 non-null   float64
 17   pf_o_int  8210 non-null   float64
 18   pf_o_fun  8201 non-null   float64
 19   pf_o_amb  8192 non-null   float64
 20   pf_o_sha  8

In [65]:
df.to_csv('cleaned.csv')